In [1]:
import polars_bio as pb
import polars as pl

In [2]:
VCZ_PATH="/Users/mwiewior/research/git/polars-bio/tmp/vcz-bench-data/30x_chr22/30x_chr22_samples10.vcz"
VCF_PATH="/Users/mwiewior/research/git/polars-bio/tmp/vcz-bench-data/30x_chr22/1kGP_high_coverage_Illumina.chr22.filtered.SNV_INDEL_SV_phased_panel.vcf.gz"

In [3]:
chrom = "chr22"
pos = 10519265
samples = ['HG00102','HG00106', 'HG00107']

In [20]:
pb.scan_vcf_zarr(VCZ_PATH).count().collect()

0rows [00:00, ?rows/s]

chrom,start,end,id,ref,alt,qual,filter,AC,AC_AFR,AC_AFR_unrel,AC_AMR,AC_AMR_unrel,AC_EAS,AC_EAS_unrel,AC_EUR,AC_EUR_unrel,AC_Het,AC_Het_AFR,AC_Het_AFR_unrel,AC_Het_AMR,AC_Het_AMR_unrel,AC_Het_EAS,AC_Het_EAS_unrel,AC_Het_EUR,AC_Het_EUR_unrel,AC_Het_SAS,AC_Het_SAS_unrel,AC_Hom,AC_Hom_AFR,AC_Hom_AFR_unrel,AC_Hom_AMR,AC_Hom_AMR_unrel,AC_Hom_EAS,AC_Hom_EAS_unrel,AC_Hom_EUR,AC_Hom_EUR_unrel,…,AN,AN_AFR,AN_AFR_unrel,AN_AMR,AN_AMR_unrel,AN_EAS,AN_EAS_unrel,AN_EUR,AN_EUR_unrel,AN_SAS,AN_SAS_unrel,CHR2,CM,END,END2,EVIDENCE,ExcHet,ExcHet_AFR,ExcHet_AMR,ExcHet_EAS,ExcHet_EUR,ExcHet_SAS,HWE,HWE_AFR,HWE_AMR,HWE_EAS,HWE_EUR,HWE_SAS,MAF_AFR_unrel,MAF_AMR_unrel,MAF_EAS_unrel,MAF_EUR_unrel,MAF_SAS_unrel,SOURCE,SVLEN,SVTYPE,genotypes
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,…,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
1066557,1066557,1066557,1066557,1066557,1066557,0,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,…,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557,1066557


In [21]:
pb.scan_vcf_zarr(VCZ_PATH, info_fields=[]).count().collect()

0rows [00:00, ?rows/s]

chrom,start,end,id,ref,alt,qual,filter,genotypes
u32,u32,u32,u32,u32,u32,u32,u32,u32
1066557,1066557,1066557,1066557,1066557,1066557,0,1066557,1066557


In [22]:
pb.scan_vcf_zarr(VCZ_PATH,samples=samples).filter((pl.col("chrom") == chrom) & (pl.col("start") == pos)).select(["chrom", "start", "id", "ref", "alt", "genotypes"]).collect()

0rows [00:00, ?rows/s]

chrom,start,id,ref,alt,genotypes
str,u32,str,str,str,struct[3]
"""chr22""",10519265,"""22:10519265:CA:C""","""CA""","""C""","{[[0, 0], [0, 0], [0, 0]],[true, true, true],[[false, false], [false, false], [false, false]]}"


In [23]:
pb.scan_vcf(VCF_PATH, samples=samples).filter((pl.col("chrom") == chrom) & (pl.col("start") == pos)).select(["chrom", "start", "id", "ref", "alt", "genotypes"]).collect()

0rows [00:00, ?rows/s]

chrom,start,id,ref,alt,genotypes
str,u32,str,str,str,struct[1]
"""chr22""",10519265,"""22:10519265:CA:C""","""CA""","""C""","{[""0|0"", ""0|0"", ""0|0""]}"


In [ ]:
pb.scan_vcf(VCF_PATH, info_fields=[]).filter((pl.col("chrom") == chrom) & (pl.col("start") == pos)).count().collect()

In [ ]:
(
    pb.scan_vcf_zarr(
        VCZ_PATH,
        info_fields=[],
        format_fields=["GT"],
    )
    .filter((pl.col("chrom") == chrom) & (pl.col("start") == pos))
    .select(["chrom", "start", "id", "ref", "alt", "genotypes"])
    .collect()
)

In [ ]:
(
    pb.scan_vcf(
        VCF_PATH,
        info_fields=[],
        format_fields=["GT"],
    )
    .filter((pl.col("chrom") == chrom) & (pl.col("start") == pos))
    .select(["chrom", "start", "id", "ref", "alt", "genotypes"])
    .collect()
)

In [4]:
%%time
# VCZ: full chromosome count by chrom
(
    pb.scan_vcf_zarr(VCZ_PATH, info_fields=[], format_fields=[])
    .group_by("chrom")
    .agg(pl.len().alias("variants"))
    .sort("chrom")
    .collect()
)

0rows [00:00, ?rows/s]

CPU times: user 110 ms, sys: 178 ms, total: 287 ms
Wall time: 210 ms


chrom,variants
str,u32
"""chr22""",1066557


In [5]:
%%time
# BGZF VCF + tabix: full chromosome count by chrom
(
    pb.scan_vcf(VCF_PATH, info_fields=[], format_fields=[])
    .group_by("chrom")
    .agg(pl.len().alias("variants"))
    .sort("chrom")
    .collect()
)

0rows [00:00, ?rows/s]

CPU times: user 6.17 s, sys: 289 ms, total: 6.45 s
Wall time: 6.35 s


chrom,variants
str,u32
"""chr22""",1066557


In [6]:
%%time
# VCZ: large-region core projection summary
region_start = 20_000_000
region_end = 20_100_000
(
    pb.scan_vcf_zarr(VCZ_PATH, info_fields=[], format_fields=[])
    .filter(
        (pl.col("chrom") == chrom)
        & (pl.col("start") >= region_start)
        & (pl.col("start") <= region_end)
    )
    .select([
        pl.len().alias("variants"),
        pl.col("start").min().alias("min_start"),
        pl.col("start").max().alias("max_start"),
    ])
    .collect()
)

0rows [00:00, ?rows/s]

CPU times: user 12.9 ms, sys: 194 ms, total: 207 ms
Wall time: 27.9 ms


variants,min_start,max_start
u32,u32,u32
2610,20000095,20099950


In [7]:
%%time
# BGZF VCF + tabix: large-region core projection summary
region_start = 20_000_000
region_end = 20_100_000
(
    pb.scan_vcf(VCF_PATH, info_fields=[], format_fields=[])
    .filter(
        (pl.col("chrom") == chrom)
        & (pl.col("start") >= region_start)
        & (pl.col("start") <= region_end)
    )
    .select([
        pl.len().alias("variants"),
        pl.col("start").min().alias("min_start"),
        pl.col("start").max().alias("max_start"),
    ])
    .collect()
)

0rows [00:00, ?rows/s]

CPU times: user 41.6 ms, sys: 9.82 ms, total: 51.5 ms
Wall time: 43 ms


variants,min_start,max_start
u32,u32,u32
2610,20000095,20099950


In [8]:
%%time
# VCZ: large-region INFO AF projection summary
region_start = 20_000_000
region_end = 20_100_000
(
    pb.scan_vcf_zarr(VCZ_PATH, info_fields=["AF"], format_fields=[])
    .filter(
        (pl.col("chrom") == chrom)
        & (pl.col("start") >= region_start)
        & (pl.col("start") <= region_end)
    )
    .select([
        pl.len().alias("variants"),
        pl.col("AF").null_count().alias("af_nulls"),
    ])
    .collect()
)

0rows [00:00, ?rows/s]

CPU times: user 14.4 ms, sys: 51.2 ms, total: 65.6 ms
Wall time: 17.1 ms


variants,af_nulls
u32,u32
2610,0


In [9]:
%%time
# BGZF VCF + tabix: large-region INFO AF projection summary
region_start = 20_000_000
region_end = 20_100_000
(
    pb.scan_vcf(VCF_PATH, info_fields=["AF"], format_fields=[])
    .filter(
        (pl.col("chrom") == chrom)
        & (pl.col("start") >= region_start)
        & (pl.col("start") <= region_end)
    )
    .select([
        pl.len().alias("variants"),
        pl.col("AF").null_count().alias("af_nulls"),
    ])
    .collect()
)

0rows [00:00, ?rows/s]

CPU times: user 50.1 ms, sys: 9.66 ms, total: 59.8 ms
Wall time: 51.1 ms


variants,af_nulls
u32,u32
2610,0


In [10]:
%%time
# VCZ: large-region GT slice for selected samples
region_start = 20_000_000
region_end = 20_100_000
(
    pb.scan_vcf_zarr(
        VCZ_PATH,
        info_fields=[],
        format_fields=["GT"],
        samples=samples,
    )
    .filter(
        (pl.col("chrom") == chrom)
        & (pl.col("start") >= region_start)
        & (pl.col("start") <= region_end)
    )
    .select(["chrom", "start", "genotypes"])
    .head(5)
    .collect()
)

0rows [00:00, ?rows/s]

CPU times: user 12.2 ms, sys: 195 ms, total: 207 ms
Wall time: 33.1 ms


chrom,start,genotypes
str,u32,struct[3]
"""chr22""",20000095,"{[[0, 0], [0, 0], [0, 0]],[true, true, true],[[false, false], [false, false], [false, false]]}"
"""chr22""",20000208,"{[[0, 0], [0, 0], [0, 0]],[true, true, true],[[false, false], [false, false], [false, false]]}"
"""chr22""",20000222,"{[[0, 0], [0, 0], [0, 0]],[true, true, true],[[false, false], [false, false], [false, false]]}"
"""chr22""",20000241,"{[[1, 1], [1, 1], [1, 1]],[true, true, true],[[false, false], [false, false], [false, false]]}"
"""chr22""",20000258,"{[[0, 0], [0, 0], [0, 0]],[true, true, true],[[false, false], [false, false], [false, false]]}"


In [11]:
%%time
# BGZF VCF + tabix: large-region GT slice for selected samples
region_start = 20_000_000
region_end = 20_100_000
(
    pb.scan_vcf(
        VCF_PATH,
        info_fields=[],
        format_fields=["GT"],
        samples=samples,
    )
    .filter(
        (pl.col("chrom") == chrom)
        & (pl.col("start") >= region_start)
        & (pl.col("start") <= region_end)
    )
    .select(["chrom", "start", "genotypes"])
    .head(5)
    .collect()
)

0rows [00:00, ?rows/s]

CPU times: user 38.9 ms, sys: 9.95 ms, total: 48.9 ms
Wall time: 40.9 ms


chrom,start,genotypes
str,u32,struct[1]
"""chr22""",20000095,"{[""0|0"", ""0|0"", ""0|0""]}"
"""chr22""",20000208,"{[""0|0"", ""0|0"", ""0|0""]}"
"""chr22""",20000222,"{[""0|0"", ""0|0"", ""0|0""]}"
"""chr22""",20000241,"{[""1|1"", ""1|1"", ""1|1""]}"
"""chr22""",20000258,"{[""0|0"", ""0|0"", ""0|0""]}"


In [12]:
%%time
# VCZ: large-region GT materialization summary for all samples
region_start = 20_000_000
region_end = 20_100_000
(
    pb.scan_vcf_zarr(VCZ_PATH, info_fields=[], format_fields=["GT"])
    .filter(
        (pl.col("chrom") == chrom)
        & (pl.col("start") >= region_start)
        & (pl.col("start") <= region_end)
    )
    .select([
        pl.len().alias("variants"),
        pl.col("genotypes").struct.field("GT").list.len().max().alias("max_samples"),
    ])
    .collect()
)

0rows [00:00, ?rows/s]

CPU times: user 204 ms, sys: 326 ms, total: 529 ms
Wall time: 716 ms


variants,max_samples
u32,u32
2610,3202


In [13]:
%%time
# BGZF VCF + tabix: large-region GT materialization summary for all samples
region_start = 20_000_000
region_end = 20_100_000
(
    pb.scan_vcf(VCF_PATH, info_fields=[], format_fields=["GT"])
    .filter(
        (pl.col("chrom") == chrom)
        & (pl.col("start") >= region_start)
        & (pl.col("start") <= region_end)
    )
    .select([
        pl.len().alias("variants"),
        pl.col("genotypes").struct.field("GT").list.len().max().alias("max_samples"),
    ])
    .collect()
)

0rows [00:00, ?rows/s]

CPU times: user 986 ms, sys: 29.8 ms, total: 1.02 s
Wall time: 965 ms


variants,max_samples
u32,u32
2610,3202


In [14]:
%%time
# VCZ: full-scan AF threshold analytics
(
    pb.scan_vcf_zarr(VCZ_PATH, info_fields=["AF"], format_fields=[])
    .with_columns(pl.col("AF").list.first().alias("af"))
    .filter(pl.col("af") >= 0.05)
    .group_by("chrom")
    .agg([
        pl.len().alias("variants_af_ge_0_05"),
        pl.col("af").min().alias("min_af"),
        pl.col("af").max().alias("max_af"),
    ])
    .sort("chrom")
    .collect()
)

0rows [00:00, ?rows/s]

CPU times: user 122 ms, sys: 276 ms, total: 397 ms
Wall time: 269 ms


chrom,variants_af_ge_0_05,min_af,max_af
str,u32,f32,f32
"""chr22""",131624,0.050125,0.999688


In [15]:
%%time
# BGZF VCF + tabix: full-scan AF threshold analytics
(
    pb.scan_vcf(VCF_PATH, info_fields=["AF"], format_fields=[])
    .with_columns(pl.col("AF").list.first().alias("af"))
    .filter(pl.col("af") >= 0.05)
    .group_by("chrom")
    .agg([
        pl.len().alias("variants_af_ge_0_05"),
        pl.col("af").min().alias("min_af"),
        pl.col("af").max().alias("max_af"),
    ])
    .sort("chrom")
    .collect()
)

0rows [00:00, ?rows/s]

CPU times: user 9.3 s, sys: 316 ms, total: 9.61 s
Wall time: 9.47 s


chrom,variants_af_ge_0_05,min_af,max_af
str,u32,f32,f32
"""chr22""",131624,0.050125,0.999688


In [16]:
%%time
# VCZ: full-scan selected-sample GT grouping
(
    pb.scan_vcf_zarr(VCZ_PATH, info_fields=[], format_fields=["GT"],samples=samples)
    .select(pl.col("genotypes").struct.field("GT").alias("GT"))
    .explode("GT")
    .group_by("GT")
    .agg(pl.len().alias("sample_calls"))
    .with_columns(pl.lit(len(samples)).alias("selected_samples"))
    .sort("GT")
    .collect()
)

0rows [00:00, ?rows/s]

CPU times: user 455 ms, sys: 352 ms, total: 806 ms
Wall time: 928 ms


GT,sample_calls,selected_samples
list[i8],u32,i32
"[0, 0]",3019615,3
"[0, 1]",59570,3
"[1, 0]",61608,3
"[1, 1]",58878,3


In [17]:
%%time
# BGZF VCF + tabix: full-scan selected-sample GT grouping
(
    pb.scan_vcf(VCF_PATH, info_fields=[], format_fields=["GT"], samples=samples)
    .select(pl.col("genotypes").struct.field("GT").alias("GT"))
    .explode("GT")
    .group_by("GT")
    .agg(pl.len().alias("sample_calls"))
    .with_columns(pl.lit(len(samples)).alias("selected_samples"))
    .sort("GT")
    .collect()
)

0rows [00:00, ?rows/s]

CPU times: user 6.67 s, sys: 286 ms, total: 6.96 s
Wall time: 6.79 s


GT,sample_calls,selected_samples
str,u32,i32
"""0|0""",3019615,3
"""0|1""",59570,3
"""1|0""",61608,3
"""1|1""",58878,3


## Full table materialization

These cells materialize the complete 1,066,557-row table into memory for VCZ and bgzipped VCF. They intentionally keep the full default column set, including INFO and genotype data.

In [18]:
import time

# VCZ: full table scan materialized in memory
t0 = time.perf_counter()
_full_vcz = pb.scan_vcf_zarr(VCZ_PATH).collect()
elapsed = time.perf_counter() - t0
vcz_full_table_scan = pl.DataFrame({
    "source": ["VCZ"],
    "rows": [_full_vcz.height],
    "columns": [_full_vcz.width],
    "wall_seconds": [elapsed],
    "estimated_size_mb": [_full_vcz.estimated_size("mb")],
})
del _full_vcz
vcz_full_table_scan

0rows [00:00, ?rows/s]

source,rows,columns,wall_seconds,estimated_size_mb
str,i64,i64,f64,f64
"""VCZ""",1066557,90,156.809024,60662.800395


In [19]:
import time

# BGZF VCF + tabix: full table scan materialized in memory
t0 = time.perf_counter()
_full_vcf = pb.scan_vcf(VCF_PATH).collect()
elapsed = time.perf_counter() - t0
vcf_full_table_scan = pl.DataFrame({
    "source": ["BGZF VCF"],
    "rows": [_full_vcf.height],
    "columns": [_full_vcf.width],
    "wall_seconds": [elapsed],
    "estimated_size_mb": [_full_vcf.estimated_size("mb")],
})
del _full_vcf
vcf_full_table_scan

0rows [00:00, ?rows/s]

source,rows,columns,wall_seconds,estimated_size_mb
str,i64,i64,f64,f64
"""BGZF VCF""",1066557,90,386.973492,10641.649911


In [ ]:
def format_gt(calls, masks, phased):
    out = []
    for alleles, allele_masks, is_phased in zip(calls, masks, phased):
        sep = "|" if bool(is_phased) else "/"
        out.append(
            sep.join("." if bool(m) or int(a) < 0 else str(int(a))
                     for a, m in zip(alleles, allele_masks))
        )
    return out

In [ ]:
import numpy as np
import zarr


In [ ]:
root = zarr.open(str(VCZ_PATH), mode="r")

wanted_samples = ["HG00102", "HG00106", "HG00107"]

sample_ids = [str(x) for x in root["sample_id"][:]]
sample_idx = [sample_ids.index(s) for s in wanted_samples]

contig_ids = np.array([str(x) for x in root["contig_id"][:]], dtype=object)
variant_idx = np.flatnonzero(
    (root["variant_position"][:] == pos)
    & (contig_ids[root["variant_contig"][:]] == chrom)
)[0]

calls = root["call_genotype"].get_orthogonal_selection(
    ([variant_idx], sample_idx, slice(None))
)[0]
masks = root["call_genotype_mask"].get_orthogonal_selection(
    ([variant_idx], sample_idx, slice(None))
)[0]
phased = root["call_genotype_phased"].get_orthogonal_selection(
    ([variant_idx], sample_idx)
)[0]

gt = format_gt(calls, masks, phased)

zarr_df = pl.DataFrame([{
    "chrom": chrom,
    "start": int(root["variant_position"][variant_idx]),
    "id": str(root["variant_id"][variant_idx]),
    "ref": str(root["variant_allele"][variant_idx, 0]),
    "alt": str(root["variant_allele"][variant_idx, 1]),
    "genotypes": {"GT": gt},
}])

zarr_df

In [ ]:
import numpy as np
import sgkit as sg



In [ ]:
needed = {
    "sample_id",
    "contig_id",
    "variant_position",
    "variant_contig",
    "variant_id",
    "variant_allele",
    "call_genotype",
    "call_genotype_mask",
    "call_genotype_phased",
}

root = zarr.open(str(VCZ_PATH), mode="r")
drop = [k for k in root.array_keys() if k not in needed]

ds = sg.load_dataset(
    str(VCZ_PATH),
    drop_variables=drop,
    consolidated=True,
)
wanted_samples = ["HG00102", "HG00106", "HG00107"]

sample_ids = [str(x) for x in ds["sample_id"].values]
sample_idx = [sample_ids.index(s) for s in wanted_samples]

contig_ids = np.array([str(x) for x in ds["contig_id"].values], dtype=object)
variant_idx = np.flatnonzero(
    (ds["variant_position"].values == pos)
    & (contig_ids[ds["variant_contig"].values] == chrom)
)[0]

sub = ds.isel(variants=[variant_idx], samples=sample_idx)

gt = format_gt(
    sub["call_genotype"].values[0],
    sub["call_genotype_mask"].values[0],
    sub["call_genotype_phased"].values[0],
)

sgkit_df = pl.DataFrame([{
    "chrom": chrom,
    "start": int(sub["variant_position"].values[0]),
    "id": str(sub["variant_id"].values[0]),
    "ref": str(sub["variant_allele"].values[0, 0]),
    "alt": str(sub["variant_allele"].values[0, 1]),
    "genotypes": {"GT": gt},
}])

sgkit_df

In [ ]:
import zarr
import sgkit as sg

